<a href="https://colab.research.google.com/github/amyylxn/Ontario-Price-Forecasting/blob/main/price_forecasting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import glob
import pandas as pd

# Only plain year files (no _v suffix, and not the rolling "most recent" file)
files = sorted(glob.glob(f'{DATA_DIR}/ieso/PUB_PriceHOEPPredispOR_[0-9][0-9][0-9][0-9].csv'))
files = [f for f in files if 2016 <= int(f[-8:-4]) <= 2025]

frames = []
for f in files:
    year = f[-8:-4]
    d = pd.read_csv(f, skiprows=3, usecols=['Date', 'Hour', 'HOEP'])
    print(year, d.shape)
    frames.append(d)

price = pd.concat(frames, ignore_index=True)
price['Date'] = pd.to_datetime(price['Date'])
price['timestamp'] = price['Date'] + pd.to_timedelta(price['Hour'] - 1, unit='h')
price = price.sort_values('timestamp').reset_index(drop=True)

print(price.shape)
price.head()

2016 (8784, 3)
2017 (8760, 3)
2018 (8760, 3)
2019 (8760, 3)
2020 (8784, 3)
2021 (8760, 3)
2022 (8760, 3)
2023 (8760, 3)
2024 (8784, 3)
2025 (2880, 3)
(81792, 4)


,Date,Hour,HOEP,timestamp
0,2016-01-01,1,-1.09,2016-01-01 00:00:00
1,2016-01-01,2,-2.41,2016-01-01 01:00:00
2,2016-01-01,3,-4.23,2016-01-01 02:00:00
3,2016-01-01,4,-4.4,2016-01-01 03:00:00
4,2016-01-01,5,-4.44,2016-01-01 04:00:00


In [7]:
# Duplicates and missing values
print(price['timestamp'].duplicated().sum(), "duplicate timestamps")
print(price['HOEP'].isna().sum(), "missing HOEP values")

# Gaps: every step between rows should be exactly 1 hour
gaps = price['timestamp'].diff().dropna()
print(gaps.value_counts())

# Overall range
print(price['timestamp'].min(), "to", price['timestamp'].max())

0 duplicate timestamps
0 missing HOEP values
timestamp
0 days 01:00:00    81791
Name: count, dtype: int64
2016-01-01 00:00:00 to 2025-04-30 23:00:00


In [8]:
path = f'{DATA_DIR}/ieso/PUB_Demand_2022.csv'
with open(path) as f:
    for _ in range(8):
        print(f.readline().rstrip())

\\Hourly Demand Report,,,
\\Created at 2023-01-31 07:30:13,,,
\\For 2022,,,
Date,Hour,Market Demand,Ontario Demand
2022-01-01,1,15150,13543
2022-01-01,2,15344,13254
2022-01-01,3,14686,12683
2022-01-01,4,14400,12491


In [9]:
dem = pd.read_csv(path, skiprows=3)
print(dem.shape)

dem['Date'] = pd.to_datetime(dem['Date'])
dem['timestamp'] = dem['Date'] + pd.to_timedelta(dem['Hour'] - 1, unit='h')

dem.info()
print(dem[['Market Demand', 'Ontario Demand']].describe())
print(dem['timestamp'].duplicated().sum(), "duplicate timestamps")
print(dem['timestamp'].diff().dropna().value_counts())

(8760, 4)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   Date            8760 non-null   datetime64[ns]
 1   Hour            8760 non-null   int64         
 2   Market Demand   8760 non-null   int64         
 3   Ontario Demand  8760 non-null   int64         
 4   timestamp       8760 non-null   datetime64[ns]
dtypes: datetime64[ns](2), int64(3)
memory usage: 342.3 KB
       Market Demand  Ontario Demand
count    8760.000000     8760.000000
mean    17647.694977    15704.322489
std      2385.870197     2309.608268
min     11675.000000    10485.000000
25%     15853.750000    13997.000000
50%     17362.000000    15561.000000
75%     19313.250000    17414.250000
max     24702.000000    22607.000000
0 duplicate timestamps
timestamp
0 days 01:00:00    8759
Name: count, dtype: int64


In [11]:
files = sorted(glob.glob(f'{DATA_DIR}/ieso/PUB_Demand_[0-9][0-9][0-9][0-9].csv'))
files = [f for f in files if 2016 <= int(f[-8:-4]) <= 2025]

frames = []
for f in files:
    d = pd.read_csv(f, skiprows=3, usecols=['Date', 'Hour', 'Ontario Demand'])
    print(f[-8:-4], d.shape)
    frames.append(d)

demand = pd.concat(frames, ignore_index=True)
demand['Date'] = pd.to_datetime(demand['Date'])
demand['timestamp'] = demand['Date'] + pd.to_timedelta(demand['Hour'] - 1, unit='h')
demand = demand[['timestamp', 'Ontario Demand']].sort_values('timestamp')

# Join onto the price table; a left join keeps your price range (ends April 30, 2025)
data = price.merge(demand, on='timestamp', how='left')

print(data.shape)
print(data['Ontario Demand'].isna().sum(), "missing demand values")
data.head()

2016 (8784, 3)
2017 (8760, 3)
2018 (8760, 3)
2019 (8760, 3)
2020 (8784, 3)
2021 (8760, 3)
2022 (8760, 3)
2023 (8760, 3)
2024 (8784, 3)
2025 (8759, 3)
(81792, 5)
0 missing demand values


,Date,Hour,HOEP,timestamp,Ontario Demand
0,2016-01-01,1,-1.09,2016-01-01 00:00:00,13417
1,2016-01-01,2,-2.41,2016-01-01 01:00:00,12968
2,2016-01-01,3,-4.23,2016-01-01 02:00:00,12395
3,2016-01-01,4,-4.4,2016-01-01 03:00:00,12228
4,2016-01-01,5,-4.44,2016-01-01 04:00:00,12116


In [12]:
import requests, os

url = 'https://reports-public.ieso.ca/public/GenOutputbyFuelHourly/PUB_GenOutputbyFuelHourly_2022.xml'
dest = f'{DATA_DIR}/ieso/PUB_GenOutputbyFuelHourly_2022.xml'

r = requests.get(url, timeout=120)
print(r.status_code, len(r.content) / 1e6, "MB")

with open(dest, 'wb') as f:
    f.write(r.content)

200 7.092996 MB


In [13]:
dest = f'{DATA_DIR}/ieso/PUB_GenOutputbyFuelHourly_2022.xml'
with open(dest) as f:
    print(f.read(3000))

<?xml version="1.0" encoding="UTF-8"?>
<?xml-stylesheet type="text/xsl" href="https://reports-public.ieso.ca/docrefs/stylesheet/GenOutputbyFuelHourly_HTML_t1-1.xsl" ?>
<Document docID="GenOutputbyFuelHourly" xmlns="http://www.ieso.ca/schema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.ieso.ca/schema https://reports-public.ieso.ca/docrefs/schema/GenOutputbyFuelHourly_r1.xsd">
<DocHeader>
<DocTitle>Generator Output by Fuel Type Hourly Report</DocTitle>
<DocRevision>1</DocRevision>
<DocConfidentiality>
<DocConfClass>PUB</DocConfClass>
</DocConfidentiality>
<CreatedAt>2023-01-01T06:50:41</CreatedAt>
</DocHeader>
<DocBody>
<DeliveryYear>2022</DeliveryYear>
<DailyData>
<Day>2022-01-01</Day>
<HourlyData>
<Hour>1</Hour>
<FuelTotal>
<Fuel>NUCLEAR</Fuel>
<EnergyValue>
<OutputQuality>0</OutputQuality>
<Output>10474</Output>
</EnergyValue>
</FuelTotal>
<FuelTotal>
<Fuel>GAS</Fuel>
<EnergyValue>
<OutputQuality>-3</OutputQuality>
<Output>163</Output>
</Energy

In [15]:
import xml.etree.ElementTree as ET
import pandas as pd

ns = {'i': 'http://www.ieso.ca/schema'}

def parse_gen(path):
    root = ET.parse(path).getroot()
    rows = []
    for day in root.findall('.//i:DailyData', ns):
        date = day.find('i:Day', ns).text
        for hr in day.findall('i:HourlyData', ns):
            hour = int(hr.find('i:Hour', ns).text)
            for ft in hr.findall('i:FuelTotal', ns):
                fuel = ft.find('i:Fuel', ns).text
                out = ft.find('.//i:Output', ns)
                rows.append((date, hour, fuel, float(out.text) if out is not None and out.text else None))
    long = pd.DataFrame(rows, columns=['Date', 'Hour', 'Fuel', 'Output'])
    wide = long.pivot_table(index=['Date', 'Hour'], columns='Fuel', values='Output').reset_index()
    wide['Date'] = pd.to_datetime(wide['Date'])
    wide['timestamp'] = wide['Date'] + pd.to_timedelta(wide['Hour'] - 1, unit='h')
    return wide.sort_values('timestamp').reset_index(drop=True)

gen = parse_gen(f'{DATA_DIR}/ieso/PUB_GenOutputbyFuelHourly_2022.xml')
print(gen.shape)
print(gen.isna().sum())
print(gen.describe())
print(gen['timestamp'].duplicated().sum(), "duplicate timestamps")
print(gen['timestamp'].diff().dropna().value_counts())
gen.head()

(8760, 9)
Fuel
Date         0
Hour         0
BIOFUEL      0
GAS          0
HYDRO        0
NUCLEAR      0
SOLAR        0
WIND         0
timestamp    0
dtype: int64
Fuel                  Date         Hour      BIOFUEL          GAS  \
count                 8760  8760.000000  8760.000000  8760.000000   
mean   2022-07-02 00:00:00    12.500000    29.804338  1799.176598   
min    2022-01-01 00:00:00     1.000000     0.000000    68.000000   
25%    2022-04-02 00:00:00     6.750000     0.000000   520.750000   
50%    2022-07-02 00:00:00    12.500000    21.000000  1459.000000   
75%    2022-10-01 00:00:00    18.250000    48.000000  2740.000000   
max    2022-12-31 00:00:00    24.000000   212.000000  7107.000000   
std                    NaN     6.922582    31.621514  1480.355396   

Fuel         HYDRO       NUCLEAR        SOLAR         WIND  \
count  8760.000000   8760.000000  8760.000000  8760.000000   
mean   4282.625000   8933.099886    79.424658  1583.202511   
min    2573.000000   4931.000

Fuel,Date,Hour,BIOFUEL,GAS,HYDRO,NUCLEAR,SOLAR,WIND,timestamp
0,2022-01-01,1,5.0,163.0,3872.0,10474.0,0.0,458.0,2022-01-01 00:00:00
1,2022-01-01,2,0.0,202.0,4076.0,10480.0,0.0,300.0,2022-01-01 01:00:00
2,2022-01-01,3,0.0,183.0,3570.0,10481.0,0.0,320.0,2022-01-01 02:00:00
3,2022-01-01,4,1.0,123.0,3332.0,10483.0,0.0,348.0,2022-01-01 03:00:00
4,2022-01-01,5,1.0,123.0,3067.0,10485.0,0.0,267.0,2022-01-01 04:00:00


In [16]:
import os, requests

BASE = 'https://reports-public.ieso.ca/public/GenOutputbyFuelHourly'
frames = []

for year in range(2016, 2026):
    dest = f'{DATA_DIR}/ieso/PUB_GenOutputbyFuelHourly_{year}.xml'
    if not os.path.exists(dest):                      # skip files you already have
        r = requests.get(f'{BASE}/PUB_GenOutputbyFuelHourly_{year}.xml', timeout=120)
        r.raise_for_status()
        with open(dest, 'wb') as f:
            f.write(r.content)
    g = parse_gen(dest)
    print(year, g.shape)
    frames.append(g)

gen_all = pd.concat(frames, ignore_index=True)
gen_all = gen_all.drop(columns=['Date', 'Hour']).sort_values('timestamp')
gen_all = gen_all.rename(columns={c: f'gen_{c.lower()}' for c in gen_all.columns if c != 'timestamp'})

data = data.merge(gen_all, on='timestamp', how='left')

print(data.shape)
print(data.isna().sum())
data.head()

2016 (8784, 9)
2017 (8760, 9)
2018 (8760, 9)
2019 (8760, 9)
2020 (8784, 9)
2021 (8760, 9)
2022 (8760, 9)
2023 (8760, 9)
2024 (8784, 9)
2025 (8760, 10)
(81792, 12)
Date                  0
Hour                  0
HOEP                  0
timestamp             0
Ontario Demand        0
gen_biofuel           0
gen_gas               0
gen_hydro             0
gen_nuclear           0
gen_solar             0
gen_wind              0
gen_other         81114
dtype: int64


,Date,Hour,HOEP,timestamp,Ontario Demand,gen_biofuel,gen_gas,gen_hydro,gen_nuclear,gen_solar,gen_wind,gen_other
0,2016-01-01,1,-1.09,2016-01-01 00:00:00,13417,3.0,919.0,2994.0,11188.0,0.0,1255.0,NaN
1,2016-01-01,2,-2.41,2016-01-01 01:00:00,12968,3.0,918.0,2999.0,11021.0,0.0,1483.0,NaN
2,2016-01-01,3,-4.23,2016-01-01 02:00:00,12395,8.0,920.0,3012.0,10904.0,0.0,1133.0,NaN
3,2016-01-01,4,-4.4,2016-01-01 03:00:00,12228,7.0,922.0,3005.0,10425.0,0.0,1347.0,NaN
4,2016-01-01,5,-4.44,2016-01-01 04:00:00,12116,2.0,921.0,3030.0,10411.0,0.0,1176.0,NaN


In [17]:
first = data.loc[data['gen_other'].notna(), 'timestamp'].min()
print("gen_other starts at:", first)
print(data.loc[data['gen_other'].notna(), 'gen_other'].describe())

data = data.drop(columns=['gen_other'])
print(data.shape)
print(data.isna().sum().sum(), "total missing values")

gen_other starts at: 2025-04-02 13:00:00
count    678.000000
mean       2.893805
std       22.900090
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      252.000000
Name: gen_other, dtype: float64
(81792, 11)
0 total missing values
